In [0]:
from pyspark.sql import functions as F, Window

bronze = spark.table("olist.bronze.orders")

# keep the latest ingested row per order_id
latest = Window.partitionBy("order_id").orderBy(F.col("_ingested_at").desc())

silver_orders = (bronze
    .withColumn("_rn", F.row_number().over(latest))
    .filter("_rn = 1")
    .drop("_rn")
    .select(
        F.col("order_id"),
        F.col("customer_id"),
        F.lower(F.trim("order_status")).alias("order_status"),
        F.to_timestamp("order_purchase_timestamp").alias("purchased_at"),
        F.to_timestamp("order_approved_at").alias("approved_at"),
        F.to_timestamp("order_delivered_carrier_date").alias("shipped_at"),
        F.to_timestamp("order_delivered_customer_date").alias("delivered_at"),
        F.to_timestamp("order_estimated_delivery_date").alias("promised_at"),
        F.col("_source_file"),
        F.col("_ingested_at"),
    )
    # the metrics the whole project hangs on
    .withColumn("delivery_days",  F.datediff("delivered_at", "purchased_at"))
    .withColumn("approval_hours", F.round(
        (F.unix_timestamp("approved_at") - F.unix_timestamp("purchased_at")) / 3600, 1))
    .withColumn("days_late",      F.datediff("delivered_at", "promised_at"))
    .withColumn("is_late",        F.col("days_late") > 0)
    .withColumn("is_delivered",   F.col("order_status") == "delivered")
)

(silver_orders.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist.silver.orders"))

In [0]:
silver_items = (spark.table("olist.bronze.order_items")
    .select(
        "order_id",
        F.col("order_item_id").cast("int").alias("item_seq"),
        "product_id",
        "seller_id",
        F.to_timestamp("shipping_limit_date").alias("ship_by"),
        F.col("price").cast("decimal(10,2)").alias("price"),
        F.col("freight_value").cast("decimal(10,2)").alias("freight"),
    )
    .withColumn("item_total", F.col("price") + F.col("freight"))
    .dropDuplicates(["order_id", "item_seq"]))

silver_items.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("olist.silver.order_items")

In [0]:
silver_customers = (spark.table("olist.bronze.customers")
    .select(
        "customer_id",
        F.col("customer_unique_id").alias("customer_key"),
        F.col("customer_zip_code_prefix").cast("int").alias("zip_prefix"),
        F.initcap(F.trim("customer_city")).alias("city"),
        F.upper(F.trim("customer_state")).alias("state"))
    .dropDuplicates(["customer_id"]))

silver_sellers = (spark.table("olist.bronze.sellers")
    .select(
        "seller_id",
        F.col("seller_zip_code_prefix").cast("int").alias("zip_prefix"),
        F.initcap(F.trim("seller_city")).alias("city"),
        F.upper(F.trim("seller_state")).alias("state"))
    .dropDuplicates(["seller_id"]))

# products need the English category names joined in
products = spark.table("olist.bronze.products")
xlat     = spark.table("olist.bronze.category_translation")

silver_products = (products.alias("p")
    .join(xlat.alias("x"),
          F.col("p.product_category_name") == F.col("x.product_category_name"),
          "left")
    .select(
        F.col("p.product_id"),
        F.coalesce(
            F.col("x.product_category_name_english"),
            F.col("p.product_category_name"),
            F.lit("unknown")
        ).alias("category"),
        F.col("p.product_weight_g").cast("int").alias("weight_g"),
        (F.col("p.product_length_cm").cast("double") *
         F.col("p.product_height_cm").cast("double") *
         F.col("p.product_width_cm").cast("double")).alias("volume_cm3"))
    .dropDuplicates(["product_id"]))

for df, name in [(silver_customers, "customers"),
                 (silver_sellers,   "sellers"),
                 (silver_products,  "products")]:
    df.write.mode("overwrite").option("overwriteSchema", "true") \
      .saveAsTable(f"olist.silver.{name}")

In [0]:
silver_reviews = (spark.table("olist.bronze.order_reviews")
    .select(
        "review_id", "order_id",
        F.col("review_score").cast("int").alias("score"),
        F.to_timestamp("review_creation_date").alias("created_at"),
        F.to_timestamp("review_answer_timestamp").alias("answered_at"))
    .filter(F.col("score").between(1, 5))
    .dropDuplicates(["order_id"]))   # a handful of orders carry two reviews

silver_reviews.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("olist.silver.reviews")

In [0]:
%sql
OPTIMIZE olist.silver.orders;
OPTIMIZE olist.silver.order_items;